# Polish GRUs (landcover) fraction based on a give threshold values [0.1%, 1%, 2%, 5%]

A function designed to combine or consolidate GRU fraction values by applying specified minimum threshold criteria. Specifically, any individual GRU fraction that falls below these defined threshold values will regrouped and summed into dominant GRUs, while fractions meeting or exceeding the threshold remain separate. It's also better to use this script after the "3-specific" step's outputs.

In [1]:
# GRUS aggregation function: polish small GRUs fractions for computation efficiency 
# load the required library
import pandas as pd
import numpy as np

# define the function
def aggregate_grus_fraction(input_grus, nonveg_grus, glacier_grus, key_id, startcolname, 
                           minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction):
    """
    Normalize and adjust GRU fractions based on category thresholds for computational efficiency.
    
    Parameters:
    - input_grus (pd.DataFrame): DataFrame with GRU fraction columns.
    - nonveg_grus (list): List of non-vegetated GRU column names.
    - glacier_grus (list): List of glacier GRU column names.
    - key_id (str): Column name for sorting.
    - startcolname (str): Prefix for GRU columns.
    - minimum_gl_fraction (float): Minimum fraction threshold for glacier GRUs.
    - minimum_nonveg_fraction (float): Minimum fraction threshold for nonveg GRUs.
    - minimum_veg_fraction (float): Minimum fraction threshold for veg GRUs.
    
    Returns:
    - pd.DataFrame: Adjusted DataFrame with normalized GRU fractions.
    """
    # Validate inputs
    if not isinstance(input_grus, pd.DataFrame):
        raise ValueError("input_grus must be a pandas DataFrame")
    if key_id not in input_grus.columns:
        print(f"key_id '{key_id}' not found in input_grus columns")
    if not all(isinstance(x, (int, float)) and x >= 0 for x in 
               [minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction]):
        raise ValueError("Minimum fractions must be non-negative numbers")
        
   # Sort by key_id
    if key_id in input_grus.columns:
        input_grus = input_grus.sort_values(by=key_id).reset_index(drop=True)
    
    # Identify all GRU columns
    all_grus = [col for col in input_grus.columns if col.startswith(startcolname)]
    if not all_grus:
        raise ValueError(f"No columns found starting with '{startcolname}'")
    
    # Ensure mutually exclusive categories
    glacier_grus = [col for col in all_grus if col in (glacier_grus or [])]
    nonveg_grus = [col for col in all_grus if col in (nonveg_grus or []) and col not in glacier_grus]
    veg_grus = [col for col in all_grus if col not in (glacier_grus + nonveg_grus)]
    
    # Validate category columns
    invalid_cols = set(glacier_grus + nonveg_grus) - set(all_grus)
    if invalid_cols:
        raise ValueError(f"Invalid GRU columns: {invalid_cols}")
    
    # Normalize fractions (row sums = 1)
    row_sums = input_grus[all_grus].sum(axis=1).replace(0, 1)
    input_grus[all_grus] = input_grus[all_grus].div(row_sums, axis=0)
    
    # Calculate original category sums
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            input_grus[f'{label}_grus_sum'] = input_grus[grus].sum(axis=1)
        else:
            input_grus[f'{label}_grus_sum'] = 0.0
    
    # Apply minimum fraction thresholds
    for col in all_grus:
        threshold = (minimum_gl_fraction if col in glacier_grus else 
                     minimum_nonveg_fraction if col in nonveg_grus else 
                     minimum_veg_fraction)
        input_grus[col] = input_grus[col].where(input_grus[col] >= threshold, 0)
    
    # Calculate new category sums after thresholding
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            input_grus[f'{label}_grus_sum_new'] = input_grus[grus].sum(axis=1)
        else:
            input_grus[f'{label}_grus_sum_new'] = 0.0
    
    input_grus['all_grus_sum_new'] = input_grus[all_grus].sum(axis=1)
    
    # Renormalize fractions to preserve original category sums
    for grus, label in zip([glacier_grus, nonveg_grus, veg_grus], ['gl', 'nonveg', 'veg']):
        if grus:
            # Compute scaling factor: original_sum / new_sum
            scale = input_grus[f'{label}_grus_sum'] / input_grus[f'{label}_grus_sum_new']
            scale = scale.replace([np.inf, -np.inf], 0).fillna(1)
            # Apply scaling to individual GRUs
            input_grus[grus] = input_grus[grus].mul(scale, axis=0)
    
    # Final normalization to ensure row sums = 1
    row_sums_new = input_grus[all_grus].sum(axis=1).replace(0, 1)
    input_grus[all_grus] = input_grus[all_grus].div(row_sums_new, axis=0)
    
    # Drop intermediate columns
    cols_to_drop = [f'{label}_grus_sum' for label in ['gl', 'nonveg', 'veg']] + \
                   [f'{label}_grus_sum_new' for label in ['gl', 'nonveg', 'veg']] + \
                   ['all_grus_sum_new']
    input_grus.drop(columns=[col for col in cols_to_drop if col in input_grus.columns], 
                    inplace=True)
    return input_grus


# Function to calculate the number of positive fraction GRUs to evaluate the level of polishing 
import pandas as pd
def count_positive_fractions(df, prefix='frac_'):
    # Identify columns starting with the prefix
    frac_cols = [col for col in df.columns if col.startswith(prefix)]
    if not frac_cols:
        raise ValueError(f"No columns found starting with '{prefix}'")
    # Count values > 0 for each matching column
    counts = df[frac_cols].gt(0).sum()
    # Add total count of all positive values in these columns
    total_count = df[frac_cols].gt(0).sum().sum()
    counts['total'] = total_count
    return counts

In [2]:
## # Example use
# glacier_grus = ['frac_19']                                   # List of GRUs (Glacierd)
# nonveg_grus = ['frac_16', 'frac_17', 'frac_18', 'frac_19']   # List of GRUs (Wetland, Water, Urban, Barrenland)
# minimum_gl_fraction=0.01
# minimum_veg_fraction=0.05 
# minimum_nonveg_fraction=0.01
# startcolname = 'frac_'
# key_id = 'ID_s'
# input_grus = aggregate_grus_fraction(input_grus, nonveg_grus, glacier_grus, key_id, startcolname, minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction)

# Example use:
# Count positive values in 'frac_' columns
# result = count_positive_fractions(df)
# print("Count of values > 0 in each 'frac_' column:")
# print(result)

We use the `Polish GRUs` Python script to build a `MESH` model setup for the Canada and Transboundary River Basin.

frac_0,  0   'mostly water'
frac_1,  1   'Temperate or sub-polar needleleaf forest'
frac_2,  2   'Sub-polar taiga needleleaf forest'
frac_5,  3   'Temperate or sub-polar broadleaf deciduous forest'
frac_6,  4   'Mixed forest'
frac_8,  5   'Temperate or sub-polar shrubland'
frac_10, 6   'Temperate or sub-polar grassland'
frac_11, 7   'Sub-polar or polar shrubland-lichen-moss'
frac_12, 8   'Sub-polar or polar grassland-lichen-moss'
frac_13, 9   'Sub-polar or polar barren-lichen-moss'
frac_14, 10  'Wetland'
frac_15, 11  'Cropland'
frac_16, 12  'Barren lands'
frac_17, 13  'Urban'
frac_18, 14  'Water'
frac_19  15  'Snow and Ice'
frac_20  16  'Dump'

In [3]:
## level of polishing and number of tiles 
# Case (1) = [0.01, 0.01, 0.01]; Number of tiles = 358798
# Case (2) = [0.02, 0.01, 0.01]; Number of tiles = 334408 (7% reduction)
# Case (3) = [0.05, 0.02, 0.01]; Number of tiles = 276031 (23% reduction)

In [ ]:
0 --- frac_1,
1 --- frac_2,
2 --- frac_5,
3 --- frac_6,

4 --- frac_8,
5 --- frac_10,
6 --- frac_11,
7 --- frac_12,

8 --- frac_13,
9 --- frac_14,
10 -- frac_15,
11 -- frac_16,
12 -- frac_17,
13 -- frac_18,
14 -- frac_19
15 -- frac_20

In [3]:
# Import the required library: 'netCDF', 'shutil', 'xarray' to read and re-save the NetCDF file.
import xarray as xr
import netCDF4 as nc
import shutil

# Path to the original NetCDF file
ddbnetcdf_path = '/home/zelalem/github-repos/camels_mesh_outputs/MESH_drainage_database.nc'

# Open the dataset
db = xr.open_dataset(ddbnetcdf_path)

# Copy the attribute information for later use
gru_attrs = db['GRU'].attrs.copy()
landuse_attrs = {'standard_name': 'Landuse classification name', 'units': 'dimensionless'}
#landuse_attrs = db['LandUse'].attrs.copy()

# Read and save the variables that will be replaced to local variables.
gru_frac = pd.DataFrame(db['GRU'].values)

# Add "frac_" prefix to each column name
gru_frac.columns = ['frac_' + str(col) for col in gru_frac.columns]

# Read and save the variables that will be replaced to local variables.
gru_names = ['Temperate or sub-polar needleleaf forest', 'Sub-polar taiga needleleaf forest', 'Temperate or sub-polar broadleaf deciduous forest',
             'Mixed forest', 'Temperate or sub-polar shrubland', 'Temperate or sub-polar grassland', 'Sub-polar or polar shrubland-lichen-moss', 
             'Sub-polar or polar grassland-lichen-moss', 'Sub-polar or polar barren-lichen-moss', 'Wetland', 'Cropland', 'Barren lands', 'Urban', 
             'Water', 'Snow and Ice', 'Dump']
#gru_names = db['LandUse'].values.tolist()

# Drop the fields that will be replaced from the dataset.
db = db.drop_vars(['GRU'])
#db = db.drop(['GRU', 'LandUse'])

# Save the size of the dimensions to local variables.
nsubbasin = db.sizes['subbasin']

print(gru_frac.columns)
print(gru_names)

Index(['frac_0', 'frac_1', 'frac_2', 'frac_3', 'frac_4', 'frac_5', 'frac_6',
       'frac_7', 'frac_8', 'frac_9', 'frac_10', 'frac_11', 'frac_12',
       'frac_13', 'frac_14', 'frac_15'],
      dtype='object')
['Temperate or sub-polar needleleaf forest', 'Sub-polar taiga needleleaf forest', 'Temperate or sub-polar broadleaf deciduous forest', 'Mixed forest', 'Temperate or sub-polar shrubland', 'Temperate or sub-polar grassland', 'Sub-polar or polar shrubland-lichen-moss', 'Sub-polar or polar grassland-lichen-moss', 'Sub-polar or polar barren-lichen-moss', 'Wetland', 'Cropland', 'Barren lands', 'Urban', 'Water', 'Snow and Ice', 'Dump']


We use the `Polish GRUs` Python script to build a `MESH` model setup for the CAMELS-SPAT stations in the Canada and Transboundary River Basin.

frac_0,  -   'mostly water'
frac_1,  0   'Temperate or sub-polar needleleaf forest'
frac_2,  1   'Sub-polar taiga needleleaf forest'
frac_5,  2   'Temperate or sub-polar broadleaf deciduous forest'
frac_6,  3   'Mixed forest'
frac_8,  4   'Temperate or sub-polar shrubland'
frac_10, 5   'Temperate or sub-polar grassland'
frac_11, 6   'Sub-polar or polar shrubland-lichen-moss'
frac_12, 7   'Sub-polar or polar grassland-lichen-moss'
frac_13, 8   'Sub-polar or polar barren-lichen-moss'
frac_14, 9  'Wetland'
frac_15, 10  'Cropland'
frac_16, 11  'Barren lands'
frac_17, 12  'Urban'
frac_18, 13  'Water'
frac_19  14  'Snow and Ice'
frac_20  15  'Dump'

In [8]:
## GRU polishing to reduce computation for a given level of GRUs fraction

# Specify the minimum threshold for [VegGRUs, NonVegGRUs, GlacierGRUs]
minimum_veg_fraction=0.05
minimum_nonveg_fraction=0.02
minimum_gl_fraction=0.01
startcolname = 'frac_'
key_id = 'ID_s'
re_group = 0

# Specify the [GlacierGRUs, NonVegGRUs, RemoveGRUs] columns
glacier_grus = ['frac_14']                                                 # List of GRUs (Glacier)
nonveg_grus = ['frac_09', 'frac_11', 'frac_12', 'frac_13', 'frac_15']      # List of GRUs (Wetland, Barrenlands, Urban, Water, dump)
gru_to_remove = []                                                         # List of GRUs (remove unrequired GRUs)

# Apply re-grouping here 
if re_group == 1:
    # Merge and Update name (Needleleaf Forest) then add those grouped into the list to remove GRUs
    cols_to_merge_nl = ['frac_0','frac_1']
    gru_frac['frac_0'] = gru_frac[cols_to_merge_nl].sum(axis=1)
    gru_names[1] = 'Needleleaf Forest'
    gru_to_remove = cols_to_merge_nl[1:]    

    # Merge and Update name (Low vegetation) then add those grouped into the list to remove GRUs
    cols_to_merge_lv = ['frac_4','frac_5','frac_6','frac_7']
    gru_frac['frac_4'] = gru_frac[cols_to_merge_lv].sum(axis=1)
    gru_names[4] = 'Low vegetation'
    gru_to_remove += cols_to_merge_lv[1:]

    # Merge and Update name (Low vegetation) then add those grouped into the list to remove GRUs
    cols_to_merge_sh = ['frac_6','frac_7']
    gru_frac['frac_6'] = gru_frac[cols_to_merge_lv].sum(axis=1)
    gru_names[6] = 'shrubland'
    gru_to_remove += cols_to_merge_sh[1:] 

    # Merge and Update name (Barren and Sparse Vegetation) then add those grouped into the list to remove GRUs
    cols_to_merge_bs = ['frac_8','frac_11']
    gru_frac['frac_11'] = gru_frac[cols_to_merge_bs].sum(axis=1)
    gru_names[11] = 'Barren and Sparse Vegetation'
    gru_to_remove += cols_to_merge_bs[:1]      

# Remove the unwanted or merged GRUs from both the gru_names and gru_fraction
index_gru_to_remove = [gru_frac.columns.get_loc(col) for col in gru_to_remove]
for idx in sorted(index_gru_to_remove, reverse=True):
    del gru_names[idx]
gru_frac = gru_frac.drop(columns = gru_to_remove)

# call for aggregation function 
input_grus = aggregate_grus_fraction(gru_frac, nonveg_grus, glacier_grus, key_id, startcolname, minimum_gl_fraction, minimum_nonveg_fraction, minimum_veg_fraction)

# sanity check
row_sums = input_grus.sum(axis=1)
print("Row sum sanity check — min:", np.nanmin(row_sums), ", max:", np.nanmax(row_sums))

# calculate the reduction of tiles
result = count_positive_fractions(input_grus)
print("Count of values > 0 in each 'frac_' column:")
print(result)

key_id 'ID_s' not found in input_grus columns
Row sum sanity check — min: 0.9999999999999997 , max: 1.0000000000000004
Count of values > 0 in each 'frac_' column:
frac_0      5580
frac_1       228
frac_2      1089
frac_3      2482
frac_4      3420
frac_5      2894
frac_6       206
frac_7       187
frac_8        48
frac_9      2321
frac_10     1112
frac_11     1777
frac_12     1342
frac_13     3301
frac_14      533
frac_15        0
total      26520
dtype: int64


In [9]:
# Convert pandas DataFrame back to NumPy array
new_gru_frac = input_grus.to_numpy()

# Update the revised GRU count.
NGRU = len(input_grus.columns)

# Save new names for the GRUs.
new_gru_names = np.array(gru_names, dtype=object)
 
# Define a dimension for the GRUs in the dataset.
# db.coords['NGRU'] = np.arange(1, (NGRU + 1), dtype = 'int32') 
    
# Save the updated fields to the dataset.
db['GRU'] = (['subbasin', 'NGRU'], new_gru_frac)
db['GRU'].attrs = gru_attrs
db['LandUse'] = (['NGRU'], new_gru_names)
db['LandUse'].attrs = landuse_attrs

# Ensure minimum channel length = 100 m
db['ChnlLength'] = db['ChnlLength'].where(db['ChnlLength'] >= 100, 100)

# Ensure minimum slope = 1e-10
db['ChnlSlope'] = db['ChnlSlope'].where(db['ChnlSlope'] >= 1e-10, 1e-10)

# Save the modified netcdf file.
if re_group == 1:
    ddbnetcdf_pathsave = '/home/zelalem/github-repos/camels_mesh_outputs/MESH_drainage_database_regrouped_Polish_0p05_0p02_0p01.nc'
    db.to_netcdf(ddbnetcdf_pathsave)
else:
    ddbnetcdf_pathsave = '/home/zelalem/github-repos/camels_mesh_outputs/MESH_drainage_database_Polish_0p05_0p02_0p01.nc'
    db.to_netcdf(ddbnetcdf_pathsave)

In [4]:
# Correct slope and length in the MESH_drainage_database.nc

# Path to the original NetCDF file
ddbnetcdf_path = '/home/zelalem/github-repos/camels_mesh_outputs/MESH_drainage_database.nc'
ddbnetcdf_pathsave = '/home/zelalem/github-repos/camels_mesh_outputs/MESH_drainage_database_slope_length_fix.nc'

# Open the dataset
db = xr.open_dataset(ddbnetcdf_path)

# Ensure minimum channel length = 100 m
db['ChnlLength'] = db['ChnlLength'].where(db['ChnlLength'] >= 100, 100)

# Ensure minimum slope = 1e-10
db['ChnlSlope'] = db['ChnlSlope'].where(db['ChnlSlope'] >= 1e-10, 1e-10)

# Save the modified netcdf file.
db.to_netcdf(ddbnetcdf_pathsave)

____